![image.png](https://i.imgur.com/a3uAqnb.png)

#🌦️ **Neural Networks for Tabular Data: Weather Type Classification**
---



In this lab, we will:
- Classify **weather types** using a neural network built with PyTorch
- Build a **Four-layer multiclass neural network classifier**
- Train the model on tabular weather data
- Evaluate the model’s performance on unseen data




In [ ]:
import kagglehub
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

##📊 **About The Dataset**

This is a **synthetic weather dataset** designed for **multiclass classification**.  
It contains numerical and categorical weather-related features such as **temperature, humidity, wind speed, precipitation, cloud cover, pressure, UV index, season, visibility, and location**.

The **target variable** is **Weather Type**, which classifies each sample into one of four classes: **Rainy, Sunny, Cloudy, or Snowy**.

Dataset link: https://www.kaggle.com/datasets/nikhil7280/weather-type-classification/data

### 🔹**Read Data**


In [ ]:
# Download latest version
path = kagglehub.dataset_download("nikhil7280/weather-type-classification")

print("Path to dataset files:", path)

df = pd.read_csv(f"{path}/weather_classification_data.csv")
df.head()

In [ ]:
# Check data types and structure
df.info()

### 🔹**Prepare Data**


> Before training the model, we'll prepare the data by **encoding categorical features and the target variable**, and **scaling numerical features** to ensure stable and efficient neural network training.





In [ ]:
# Encode features and target using LabelEncoder
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df

In [ ]:
# Standardize features using StandardScaler
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Weather Type")  ### DON'T SCALE THE TARGET
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df



> After preprocessing data, we'll first split the dataset into features and target labels, apply a train–test split, and then convert the data into PyTorch tensors




In [ ]:
# split features from targets
X = df.drop("Weather Type",axis=1)
y = df['Weather Type']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# transform to tensors
X_train = torch.tensor(X_train.values, dtype=torch.float32)
X_test  = torch.tensor(X_test.values, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test  = torch.tensor(y_test.values, dtype=torch.long)



> After converting the data into tensors, we group features and labels into **TensorDataset** objects.  
We then use **DataLoaders** to load the data in mini-batches for efficient training and evaluation.





In [ ]:
# Create TensorDatasets for training and testing
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

# Create Dataloaders to train and test data in batches
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Print dataset sizes
print("Train dataset:", len(train_dataset))
print("Test dataset:", len(test_dataset))

# Get the first batch from the training DataLoader
X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")

> Data now is ready for the model ! Lets build the model class.
---

### 🔹**Model Class**


Let’s create the **architecture of our model** by implementing a **four-layer neural network classifier**.

The model consists of multiple hidden layers with non-linear activations, followed by an output layer for multiclass prediction.  
Instead of **sigmoid**, we use **softmax** at the output layer:
- **Sigmoid** outputs values independently in the range (0, 1)
- **Softmax** converts a vector of scores into class probabilities that sum to 1

Softmax formula:  
$$
\text{Softmax}(x)_i = \frac{e^{x_i}}{\sum_{j} e^{x_j}}
$$


In [ ]:
class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NN4Layer, self).__init__()

        # First linear layer: input features -> hidden layer
        self.layer1 = nn.Linear(input_dim, hidden_dim)

        # Second linear layer: hidden layer -> hidden layer
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        # Third linear layer: hidden layer -> hidden layer
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

        # Output layer: hidden layer -> number of classes (logits)
        self.layer4 = nn.Linear(hidden_dim, output_dim)

        # ReLU activation for non-linearity
        self.relu = nn.ReLU()

    # Defines how input data flows through the network
    def forward(self, x):
        # First hidden layer
        a1 = self.relu(self.layer1(x))

        # Second hidden layer
        a2 = self.relu(self.layer2(a1))

        # Third hidden layer
        a3 = self.relu(self.layer3(a2))

        # Output layer (raw scores / logits)
        output = self.layer4(a3)  # we said we'll use softmax, where is it? ¯\(ツ)/¯

        return output


📌 **Note on Softmax**

The output of this model consists of **raw scores (logits)**, not probabilities.  
When using `CrossEntropyLoss`, **Softmax is applied internally**, so it should **not** be included in the model’s architecture.

Softmax can be applied **only during evaluation** if class probabilities are needed.

---

### 🔹**Training Loop**

In [ ]:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
    # Set the model to training mode
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Move batch to the selected device
        X_batch = X_batch.to(device)         # shape: (batch_size, num_features)
        y_batch = y_batch.to(device)         # shape: (batch_size,)

        # Forward pass (outputs are logits)
        outputs = model(X_batch)             # shape: (batch_size, num_classes)
        loss = criterion(outputs, y_batch)

        # Backward pass & optimization
        optimizer.zero_grad()   # Clear previous gradients
        loss.backward()         # Compute gradients
        optimizer.step()        # Update model parameters

        running_loss += loss.item()

    # Average loss over all batches
    avg_loss = running_loss / len(train_loader)

    return avg_loss


### 🔹**Validation Loop**

In [ ]:
def validate(model, criterion, test_loader, device):
    # Set the model to evaluation mode
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            # Move data to device
            X_batch = X_batch.to(device)     # shape: (batch_size, num_features)
            y_batch = y_batch.to(device)     # shape: (batch_size,)

            # Forward pass
            outputs = model(X_batch)         # shape: (batch_size, num_classes)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item()

            # Apply Softmax to get probabilities
            probabilities = F.softmax(outputs, dim=1)

            # Pick the classes with highest probabilities
            predicted = torch.argmax(probabilities, dim=1)

            # Accuracy calculation
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = running_loss / len(test_loader)
    accuracy = correct / total

    return avg_loss, accuracy

---

### 🔹**Running Training**

In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model parameters
input_dim = X_train.shape[1]   # Number of tabular features
hidden_dim = 14                # Design choice
output_dim = 4                 # Weather classes: Rainy, Sunny, Cloudy, Snowy

# Instantiate model
model = NN4Layer(input_dim, hidden_dim, output_dim).to(device)

# Print the model architecture
print("Model Architecture:\n")
print(model)

# Calculate the total number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")


In [ ]:
num_epochs = 20
learning_rate = 0.001

# Define criterion (loss function) - using CrossEntropyLoss as model now outputs logits
criterion = nn.CrossEntropyLoss()
# Define optimizer
optimizer = AdamW(model.parameters(), learning_rate)

In [ ]:
# Run Training
train_losses = []
val_losses = []
val_accuracies = []

print('Starting Training...')
for epoch in range(num_epochs):
    # Train one epoch
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

    # Validate
    val_loss, val_accuracy = validate(model, criterion, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')

print('Training Complete!')

> Lets see what train and validation losses look like

In [ ]:
# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

### 💾 **Saving the Model**

After training, we can save the model so it can be **reused later without retraining**.  
In PyTorch, the recommended approach is to save the model’s **state dictionary**, which contains all learned parameters.

In [ ]:
# Save the trained model parameters
torch.save(model.state_dict(), "weather_nn4layer.pth")

print("Model saved successfully!")

> if you want to use the model :


In [ ]:
# Recreate the model architecture
model = NN4Layer(input_dim, hidden_dim, output_dim)
# Load saved parameters
model.load_state_dict(torch.load("weather_nn4layer.pth"))
# Set model to evaluation mode
model.eval()
print("Model loaded successfully!")

## We're done!
try to play with hyperparameters and evaluate the results :)

<a href="https://imgflip.com/i/ags9pe"><img src="https://i.imgflip.com/ags9pe.jpg" width="30%"/>

### **Contributed by: Yara Alzahrani**